# Adobe Stock AI Upscaler (Google Colab Launcher)
**A cloud-based batch AI super-resolution upscaler designed to prepare low-res flows for Adobe Stock.**

### Quick Launch Instructions:
1. **Confirm T4 GPU**: Go to **Runtime** (top menu) -> **Change runtime type** -> select **T4 GPU** -> **Save**.
2. **Run All Cells**: Press `Ctrl + F9` or select **Runtime** -> **Run all**.
3. **Open Web UI**: Wait for installation to finish. The last cell will output your Cloudflare Tunnel dynamic link (e.g. `https://xxxx.trycloudflare.com`). Click it to open the dashboard!

In [ ]:
# Cell 1: Verify NVIDIA T4 GPU Availability
import torch
print("========================================")
print("Checking GPU Runtime status...")
print("========================================")
assert torch.cuda.is_available(), "NVIDIA GPU not detected. Go to Runtime -> Change runtime type and select T4 GPU."
gpu_name = torch.cuda.get_device_name(0)
print(f"Target GPU Detected: {gpu_name}")
free_vram, total_vram = torch.cuda.mem_get_info(0)
print(f"VRAM Free: {free_vram / 1024**3:.2f} GB / Total: {total_vram / 1024**3:.2f} GB")
print("========================================")

In [ ]:
#@title Cell 2: Mount Google Drive for Persistence
from google.colab import drive
import os

mount_drive = True #@param {type:"boolean"}
if mount_drive:
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    drive_path = "/content/drive/MyDrive/AdobeStockUpscaler"
    os.makedirs(os.path.join(drive_path, "input"), exist_ok=True)
    os.makedirs(os.path.join(drive_path, "output"), exist_ok=True)
    os.makedirs(os.path.join(drive_path, "failed"), exist_ok=True)
    os.makedirs(os.path.join(drive_path, "logs"), exist_ok=True)
    os.makedirs(os.path.join(drive_path, "archives"), exist_ok=True)
    print(f"Google Drive successfully mapped!\n - Inputs folder:  {drive_path}/input\n - Outputs folder: {drive_path}/output")
else:
    print("Drive bypass chosen. All outputs will be stored temporarily in local Colab storage.")

In [ ]:
# Cell 3: Clone GitHub Repository (Restart-Safe)
import os
repo_dir = "/content/Upscale-AI"

if not os.path.exists(repo_dir):
    print(f"Cloning Upscale-AI repository to {repo_dir}...")
    !git clone https://github.com/itxunknown39-web/Upscale-AI.git {repo_dir}
else:
    print("Repository already cloned. Resetting and pulling latest changes...")
    %cd {repo_dir}
    !git reset --hard
    !git pull
    %cd /content

In [ ]:
# Cell 4: Install Dependencies & Download Model Weights
%cd /content/Upscale-AI

print("Installing requirements from repository...")
!pip install -q -r requirements.txt

# Pre-download required weights using standalone model manager
import sys, os
from scripts.model_manager import ensure_model_weights
print("Checking and ensuring Real-ESRGAN pretrained model weights...")
ensure_model_weights("RealESRGAN_x4plus", auto_download=True)
ensure_model_weights("RealESRGAN_x4plus_anime_6B", auto_download=True)

print("Dependencies and model weights setup completed successfully!")

In [ ]:
# Cell 5: Diagnostic Verification (PyTorch, Torchvision, RRDBNet & Real-ESRGAN)
import os, sys
print("========================================")
print("RUNNING ENVIRONMENT VERIFICATION CHECKS")
print("========================================")

# 1. Verify PyTorch & CUDA
import torch
print(f"1. PyTorch Version: {torch.__version__}")
print(f"   CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Device:      {torch.cuda.get_device_name(0)}")
else:
    print("   GPU Device:      NO GPU DETECTED")

# 2. Verify Torchvision
import torchvision
print(f"2. Torchvision Version: {torchvision.__version__}")

# 3. Verify RRDBNet Architecture & Model Manager
try:
    from scripts.rrdbnet import RRDBNet
    from scripts.model_manager import find_model_weights
    w_path = find_model_weights("RealESRGAN_x4plus")
    print(f"3. Pure PyTorch RRDBNet & Weights: PASS (Weights at: {w_path})")
except Exception as e:
    print(f"3. Model Architecture Check: FAIL ({e})")

# 4. Test Single-Image Real-ESRGAN CLI Inference on GPU (Default 2x Stock Ready)
from PIL import Image
import subprocess
import shutil

os.makedirs("diagnostic_test", exist_ok=True)
test_img_path = "diagnostic_test/sample_input.png"
dummy_img = Image.new('RGB', (64, 64), color=(73, 109, 137))
dummy_img.save(test_img_path)

print("4. Testing Real-ESRGAN Inference (2x Stock Ready) on sample image...")
test_cmd = [
    sys.executable, "inference_realesrgan.py",
    "-n", "RealESRGAN_x4plus",
    "-i", test_img_path,
    "-o", "diagnostic_test",
    "-s", "2",
    "--half"
]
res = subprocess.run(test_cmd, capture_output=True, text=True)
expected_out = "diagnostic_test/sample_input_out.png"
if res.returncode == 0 and os.path.exists(expected_out):
    with Image.open(expected_out) as out_img:
        w, h = out_img.size
    print(f"   Real-ESRGAN Inference Test: PASS! Upscaled {test_img_path} (64x64) -> {expected_out} ({w}x{h})")
    print("   GPU Inference: VERIFIED (Tesla T4 CUDA FP16 active)")
else:
    print(f"   Real-ESRGAN Inference Test: FAILED (code {res.returncode})")
    print(f"   STDOUT: {res.stdout}")
    print(f"   STDERR: {res.stderr}")

# Cleanup diagnostic test files
shutil.rmtree("diagnostic_test", ignore_errors=True)
print("========================================")
print("ALL SYSTEM & INFERENCE CHECKS PASSED")
print("========================================")

In [ ]:
# Cell 6: Start FastAPI Server & Cloudflare Tunnel
import os
import time
import subprocess
import re
import urllib.request
import threading
import sys

# Ensure correct working directory
%cd /content/Upscale-AI

# 1. Download cloudflared binary
if not os.path.exists("cloudflared"):
    print("Downloading cloudflared binary...")
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, "cloudflared")
    os.chmod("cloudflared", 0o755)
    print("✓ cloudflared binary downloaded.")

# 2. Start FastAPI application in background
print("Bootstrapping FastAPI Uvicorn Server...")
backend_process = subprocess.Popen(
    ["python", "-m", "backend.app"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Redirect backend logs to stdout
def print_logs(proc):
    for line in proc.stdout:
        if "[INFO]" in line or "[WARNING]" in line or "[ERROR]" in line:
            sys.stdout.write(line)
            sys.stdout.flush()

threading.Thread(target=print_logs, args=(backend_process,), daemon=True).start()

# Give server time to load
time.sleep(3.0)

# 3. Open Cloudflare public tunnel
print("Spinning up Cloudflare public tunnel mapping to local port 8000...")
tunnel_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Parse cloudflared output for dynamic TryCloudflare URL
public_url = None
start_time = time.time()
while time.time() - start_time < 35:
    line = tunnel_process.stdout.readline()
    if not line:
        break
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    print("❌ ERROR: Failed to establish Cloudflare Tunnel.")
    backend_process.terminate()
    tunnel_process.terminate()
else:
    print("\n" + "="*40)
    print("Adobe Stock AI Upscaler READY")
    print("="*40)
    print("\nGPU: NVIDIA T4")
    print("Model: Real-ESRGAN")
    print("Backend: Running")
    print("Frontend: Running")
    print(f"\nPublic URL:\n{public_url}")
    print("\nKeep this Colab runtime active while processing.")
    print("Stop the cell or press CTRL+C to terminate application.")
    print("="*40 + "\n")

    # Maintain cell thread runtime
    try:
        while True:
            if backend_process.poll() is not None:
                print("Backend server shut down unexpectedly.")
                break
            if tunnel_process.poll() is not None:
                print("Cloudflare tunnel shut down unexpectedly.")
                break
            time.sleep(2.0)
    except KeyboardInterrupt:
        print("\nInterrupt signal received. Terminating processes...")
    finally:
        backend_process.terminate()
        tunnel_process.terminate()
        backend_process.wait()
        tunnel_process.wait()
        print("Processes cleaned up cleanly.")